# 나이브 베이즈 분류기

특정 문서 안에 들어 있는 단어들은 해당 문서의 주제를 추측할 수 있는 증거이다.
예를 들어 어떤 문서에 다음 단어가 많이 등장한다고 하자.

```text
nasa, orbit, launch, shuttle, space
```

사람은 이 문서가 우주 관련 글일 가능성이 높다고 판단할 수 있다.  
반대로 다음 단어가 많이 등장하면 하키 관련 글일 가능성이 높다고 판단할 수 있다.

```text
hockey, puck, team, season, goal
```

나이브 베이즈도 이와 비슷한 방식으로 동작한다.  
각 단어를 증거로 보고, 이 증거들이 어느 클래스에서 더 자주 나타났는지를 계산하여 가장 그럴듯한 클래스를 선택한다.

## 해결하려는 문제

분류 모델이 최종적으로 알고 싶은 것은 다음 확률이다.

```text
P(클래스 | 문서)
```

읽는 방식은 다음과 같다.

```text
문서가 주어졌을 때, 이 문서가 특정 클래스일 확률
```

예를 들어 뉴스 문서 분류에서는 다음과 같은 질문을 하게 된다.

```text
P(sci.space | 문서 안에 nasa, orbit, launch가 등장함)
P(rec.sport.hockey | 문서 안에 nasa, orbit, launch가 등장함)
P(talk.politics.guns | 문서 안에 nasa, orbit, launch가 등장함)
...
```

그리고 이 중 가장 큰 값을 가진 클래스를 예측 결과로 선택한다.

```text
가장 가능성이 높은 클래스 = 모델의 예측값
```

## 베이즈 정리 사용

우리가 직접 알고 싶은 것은 다음 값이다.

```text
P(클래스 | 문서)
```

그런데 이 값을 직접 계산하기는 어렵다.  
그래서 베이즈 정리를 이용해 계산하기 쉬운 형태로 바꾼다.

$$
P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}
$$

여기서 의미는 다음과 같다.

| 기호 | 의미 | 텍스트 분류 관점 |
| --- | --- | --- |
| $P(C\|X)$ | 사후 확률 | 문서 X가 주어졌을 때 클래스 C일 확률 |
| $P(X\|C)$ | 우도 | 클래스 C의 문서에서 이런 단어들이 나올 확률 |
| $P(C)$ | 사전 확률 | 전체 문서 중 클래스 C가 나올 기본 확률 |
| $P(X)$ | 증거 | 문서 X 자체가 나타날 확률 |

분류에서는 같은 문서 X를 여러 클래스에 대해 비교한다.  
따라서 $P(X)$는 모든 클래스에 공통으로 적용된다.

그래서 실제 비교에서는 다음 값이 중요하다.

```text
P(X|C) × P(C)
```

즉, 나이브 베이즈는 다음 두 가지를 함께 본다.

```text
1. 원래 그 클래스가 얼마나 자주 등장하는가?
2. 현재 문서의 단어들이 클래스 C의 학습 문서들에서 얼마나 자주 등장했던 단어들인가?
```

## 나이브의 의미

나이브 베이즈의 “나이브”는 단순하다는 뜻이다.  
이 모델은 문서 안의 단어들이 서로 독립적으로 등장한다고 가정한다.

예를 들어 다음 단어들이 있다고 하자.

```text
nasa, orbit, launch
```

현실적으로 이 단어들은 서로 관련이 있다.  
하지만 나이브 베이즈는 단순하게 다음처럼 계산한다.

```text
P(nasa, orbit, launch | sci.space)
= P(nasa | sci.space)
× P(orbit | sci.space)
× P(launch | sci.space)
```

즉, 단어 간의 관계나 어순을 깊게 이해하지 않는다.  
각 단어가 해당 클래스에서 얼마나 자주 등장했는지를 따로 계산한 뒤, 그 값들을 조합한다.

이 가정은 현실적으로 완벽하지 않다.  
하지만 텍스트 분류에서는 꽤 잘 동작한다.  
왜냐하면 많은 문서 분류 문제는 문장 구조보다 어떤 단어들이 등장했는지가 주제 판단에 큰 영향을 주기 때문이다.

## 모델 학습 단계

나이브 베이즈는 딥러닝 모델처럼 복잡한 가중치를 여러 번 업데이트하면서 학습하지 않는다.  
학습 과정은 매우 단순하다.

```text
클래스별로 문서 수를 센다.
클래스별로 단어 등장 횟수를 센다.
클래스별 단어 확률을 계산한다.
```

예를 들어 학습 데이터에서 다음과 같은 통계가 만들어졌다고 하자.

```text
sci.space 문서들:
nasa 100번, orbit 80번, launch 70번, game 2번

rec.sport.hockey 문서들:
game 120번, team 90번, score 70번, nasa 1번
```

그러면 모델은 다음과 같은 정보를 갖게 된다.

```text
nasa, orbit, launch는 sci.space에서 자주 나온다.
game, team, score는 rec.sport.hockey에서 자주 나온다.
```

이후 새 문서가 들어왔을 때, 문서 안의 단어들이 어느 클래스의 단어 분포와 더 가까운지 비교한다.

## 모델 예측 단계

새 문서가 다음과 같다고 하자.

```text
nasa launch orbit
```

모델은 모든 클래스에 대해 점수를 계산한다.

```text
sci.space 점수
= P(sci.space)
× P(nasa | sci.space)
× P(launch | sci.space)
× P(orbit | sci.space)

rec.sport.hockey 점수
= P(rec.sport.hockey)
× P(nasa | rec.sport.hockey)
× P(launch | rec.sport.hockey)
× P(orbit | rec.sport.hockey)
```

그리고 가장 큰 점수를 가진 클래스를 선택한다.

```text
가장 큰 점수를 가진 클래스 = 예측 결과
```

이 관점에서 보면 나이브 베이즈는 문서를 이해한다기보다, 단어 증거를 모아서 가장 그럴듯한 클래스를 고르는 모델이라고 볼 수 있다.

## 텍스트 분류에서 자주 사용하는 이유

텍스트 데이터는 단어 종류가 많아 feature 수가 매우 커진다.

```text
문서 수: 10,000개
단어 수: 30,000개
→ 입력 feature 수가 30,000개
```

하지만 한 문서에는 전체 단어 중 일부만 등장한다.  
즉, 벡터의 대부분은 0이다.

이런 데이터를 고차원 희소 벡터라고 한다.

나이브 베이즈는 이러한 텍스트 벡터에 잘 맞는다.  
학습 속도가 빠르고, 데이터가 아주 많지 않아도 기준 성능을 만들기 좋다.

그래서 텍스트 분류에서는 다음과 같은 흐름으로 자주 사용한다.

```text
1. CountVectorizer 또는 TfidfVectorizer로 문서를 숫자 벡터로 변환한다.
2. MultinomialNB로 빠르게 baseline 성능을 확인한다.
3. 이후 Logistic Regression, SVM, 딥러닝 모델 등과 비교한다.
```

## 한계

나이브 베이즈는 단어 간 관계를 깊게 보지 못한다.

예를 들어 다음 두 표현은 의미가 다르다.

```text
good
not good
```

하지만 단어 단위로 보면 둘 다 `good`이라는 단어를 포함한다.  
나이브 베이즈는 어순, 문맥, 부정 표현, 반어적 표현을 정교하게 이해하지 못한다.

따라서 다음과 같은 문제에서는 한계가 나타날 수 있다.

```text
문맥 이해가 중요한 감정 분석
비꼼이나 반어 표현
긴 문장 관계를 봐야 하는 분류
단어 조합의 의미가 중요한 문제
```

그래도 주제 분류처럼 특정 단어의 등장 여부와 빈도가 중요한 문제에서는 좋은 출발점이 된다.

# MultinomialNB

## MultinomialNB란?

`MultinomialNB`는 나이브 베이즈 분류기 중에서 텍스트 분류에 자주 사용하는 모델이다.

이 모델은 문서를 다음과 같은 단어 등장 횟수 벡터로 본다.

```text
문서: I love this movie

amazing film hate love movie terrible this was
0       0    0    1    1     0        1    0
```

즉, 문장의 의미를 문법적으로 해석하는 것이 아니라, 어떤 단어가 몇 번 등장했는지를 보고 분류한다.

## Multinomial이란?

Multinomial은 다항분포를 의미한다.  
다항분포는 여러 범주 중에서 어떤 항목이 몇 번 나왔는지를 다루는 확률분포이다.

예를 들어 다음 상황들이 다항분포와 연결된다.

```text
주사위를 10번 던졌을 때 각 눈이 나온 횟수
설문조사에서 각 선택지가 선택된 횟수
문서에서 각 단어가 등장한 횟수
```

텍스트 분류에서는 각 단어를 하나의 범주로 볼 수 있다.

```text
단어 범주: amazing, film, hate, love, movie, terrible, this, was
문서 벡터: 각 단어가 문서에 등장한 횟수
```

그래서 단어 빈도 기반 텍스트 분류에는 `MultinomialNB`가 잘 어울린다.

## CountVectorizer와 MultinomialNB의 연결

`CountVectorizer`는 문서를 단어 빈도 벡터로 바꾼다.  
`MultinomialNB`는 그 단어 빈도 벡터를 보고 클래스별 확률을 계산한다.

두 도구의 역할은 다음과 같이 나눌 수 있다.

```text
CountVectorizer
→ 문서를 숫자 벡터로 변환한다.

MultinomialNB
→ 그 숫자 벡터를 보고 어느 클래스일 가능성이 높은지 계산한다.
```

중요한 점은 `MultinomialNB`가 단어 자체의 뜻을 아는 것이 아니라는 점이다.  
모델은 학습 데이터에서 단어와 클래스의 관계를 빈도 기반으로 기억한다.

```text
love, amazing이 pos 문서에서 자주 나왔으면 pos에 유리한 증거가 된다.
hate, terrible이 neg 문서에서 자주 나왔으면 neg에 유리한 증거가 된다.
```

## 스무딩이 필요한 이유

어떤 단어가 특정 클래스에서 한 번도 등장하지 않으면 그 단어의 확률은 0이 된다.

```text
P(terrible | pos) = 0
```

그런데 나이브 베이즈는 여러 단어 확률을 곱해서 클래스 점수를 만든다.  
따라서 단어 하나의 확률이 0이면 전체 점수도 0이 되어 버린다.

```text
P(pos) × P(I | pos) × P(love | pos) × P(terrible | pos)
= 전체 점수 0
```

이 문제를 막기 위해 스무딩을 사용한다.

`MultinomialNB`의 기본값은 다음과 같다.

```python
MultinomialNB(alpha=1.0)
```

`alpha=1.0`은 라플라스 스무딩을 의미한다.  
각 단어가 최소한 조금은 등장한 것처럼 보정하여 확률이 0이 되는 문제를 막는다.

## 로그 확률을 사용하는 이유

단어 확률은 대부분 매우 작은 값이다.

```text
0.01 × 0.03 × 0.002 × 0.05 × ...
```

이렇게 작은 값을 계속 곱하면 컴퓨터가 너무 작은 수를 제대로 표현하지 못할 수 있다.  
그래서 실제 계산에서는 확률을 그대로 곱하지 않고 로그 확률을 더한다.

```text
log(a × b × c) = log(a) + log(b) + log(c)
```

따라서 실제 내부 계산은 다음 형태에 가깝다.

```text
log P(클래스)
+ 단어1 등장횟수 × log P(단어1 | 클래스)
+ 단어2 등장횟수 × log P(단어2 | 클래스)
+ ...
```

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 학습데이터
texts = [
    "I love this movie",       # pos
    "This film was amazing",   # pos
    "I hate this movie",       # neg
    "This movie was terrible"  # neg
]
labels = ["pos", "pos", "neg", "neg"]

# 빈도 수 기반 문서 벡터 생성
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(texts)
print(X_train.toarray())

doc_vec_df = pd.DataFrame(X_train.toarray(),columns=vectorizer.get_feature_names_out())
doc_vec_df

[[0 0 0 1 1 0 1 0]
 [1 1 0 0 0 0 1 1]
 [0 0 1 0 1 0 1 0]
 [0 0 0 0 1 1 1 1]]


,amazing,film,hate,love,movie,terrible,this,was
0,0,0,0,1,1,0,1,0
1,1,1,0,0,0,0,1,1
2,0,0,1,0,1,0,1,0
3,0,0,0,0,1,1,1,1


In [7]:
# 모델 학습
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(X_train,labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [8]:
# 예측
test_text = ['I love this movie']
X_test =vectorizer.transform(test_text)
print(X_train.toarray())
pred = clf.predict(X_test)
pred

[[0 0 0 1 1 0 1 0]
 [1 1 0 0 0 0 1 1]
 [0 0 1 0 1 0 1 0]
 [0 0 0 0 1 1 1 1]]


array(['pos'], dtype='<U3')

In [9]:
# 예측 과정
proba = clf.predict_proba(X_test)
classes = clf.classes_
print(proba)
print(classes)

[[0.42857143 0.57142857]]
['neg' 'pos']


In [12]:
# log 확률 : 특정 클래스 확률 값 + 각 토큰별 개수 + 특정 클래스의 토큰 확률

# 클래스 사전 확률: log(0.5,0.5)
class_log_prior = clf.class_log_prior_
print('클래스 사전 확률 : ',class_log_prior)

# 특정 클래스에서 토큰별 확률
feature_log_prob = clf.feature_log_prob_
print('특정 클래스에서 토큰별 확률 : \n', feature_log_prob)

pd.DataFrame(feature_log_prob, columns=vectorizer.get_feature_names_out(),index=clf.classes_)

클래스 사전 확률 :  [-0.69314718 -0.69314718]
특정 클래스에서 토큰별 확률 : 
 [[-2.7080502  -2.7080502  -2.01490302 -2.7080502  -1.60943791 -2.01490302
  -1.60943791 -2.01490302]
 [-2.01490302 -2.01490302 -2.7080502  -2.01490302 -2.01490302 -2.7080502
  -1.60943791 -2.01490302]]


,amazing,film,hate,love,movie,terrible,this,was
neg,-2.708050,-2.708050,-2.014903,-2.708050,-1.609438,-2.014903,-1.609438,-2.014903
pos,-2.014903,-2.014903,-2.708050,-2.014903,-2.014903,-2.708050,-1.609438,-2.014903


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

x_vec = X_test.toarray()[0]

# 각 클래스별 log_proba 직접 계산
log_scores = []

for class_idx, class_name in enumerate(clf.classes_):
    log_proba = class_log_prior[class_idx]  # 특정 클래스 확률

    for i, count in enumerate(x_vec):
        if count > 0:
            log_proba += count * feature_log_prob[class_idx][i]     # 특정 클래스별 토큰별 확률
        
    log_scores.append(log_proba)

print(log_scores)
proba = F.softmax(torch.tensor(log_scores),dim=0)
proba

[np.float64(-6.620073206530356), np.float64(-6.332391134078575)]


tensor([0.4286, 0.5714], dtype=torch.float64)

## 뉴스그룹 토픽 분류

```text
문서 안에 등장한 단어들이 어느 뉴스그룹에서 자주 등장했는가?
```

예를 들어 `space`, `nasa`, `orbit` 같은 단어가 많으면 `sci.space` 클래스의 점수가 높아지고,  
`hockey`, `team`, `season` 같은 단어가 많으면 `rec.sport.hockey` 클래스의 점수가 높아진다.

In [15]:
# from sklearn.datasets import fetch_20newsgroups

# newsgroup = fetch_20newsgroups(subset='all')
# print(newsgroup.data[0])
# print(newsgroup.target[0])
# print(newsgroup.target_names)

In [18]:
import zipfile
import re
import numpy as np
from sklearn.utils import Bunch
from sklearn.utils import check_random_state

zip_path = "archive.zip"

target_names = [
    "alt.atheism",
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.windows.x",
    "misc.forsale",
    "rec.autos",
    "rec.motorcycles",
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.crypt",
    "sci.electronics",
    "sci.med",
    "sci.space",
    "soc.religion.christian",
    "talk.politics.guns",
    "talk.politics.mideast",
    "talk.politics.misc",
    "talk.religion.misc",
]

data = []
target = []

with zipfile.ZipFile(zip_path, "r") as z:
    for label, name in enumerate(target_names):
        filename = f"{name}.txt"

        text = z.read(filename).decode("latin1")

        # 각 문서는 보통 From: 으로 시작하므로 From: 기준으로 분리한다.
        docs = re.split(r"(?m)(?=^From: )", text)
        docs = [doc.strip() for doc in docs if doc.strip()]

        data.extend(docs)
        target.extend([label] * len(docs))

# fetch_20newsgroups()와 비슷하게 데이터를 섞는다.
rng = check_random_state(42)
indices = np.arange(len(data))
rng.shuffle(indices)

data = [data[i] for i in indices]
target = np.array([target[i] for i in indices])

newsgroup = Bunch(
    data=data,
    target=target,
    target_names=target_names
)

print(newsgroup.data[0])
print(newsgroup.target[0])
print(newsgroup.target_names)

From: jbreed@doink.b23b.ingr.com (James B. Reed)
Subject: Re: space news from Feb 15 AW&ST

In article <C5ros0.uy@zoo.toronto.edu>, henry@zoo.toronto.edu (Henry Spencer) writes:
|> [Pluto's] atmosphere will start to freeze out around 2010, and after about
|> 2005 increasing areas of both Pluto and Charon will be in permanent
|> shadow that will make imaging and geochemical mapping impossible.

Where does the shadow come from?  There's nothing close enough to block
sunlight from hitting them.  I wouldn't expect there to be anything block
our view of them either.  What am I missing?

	Jim

Newsgroup: sci.space
Document_id: 61129
14
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politi

In [25]:
# 데이터 전처리
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = newsgroup.data
y = newsgroup.target

vectorizer = TfidfVectorizer(stop_words='english')
X_transformed = vectorizer.fit_transform(X)
print(X_transformed.shape)

(39298, 185261)


In [26]:
X_train,X_test,y_train,y_test = train_test_split(X_transformed,y,test_size=0.2,random_state=42)
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

(31438, 185261) (31438,)
(7860, 185261) (7860,)


In [27]:
# 모델 학습
model = MultinomialNB()
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [28]:
# 모델 평가
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print('accuracy : ',accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred,target_names=newsgroup.target_names))

accuracy :  0.9498727735368957
                          precision    recall  f1-score   support

             alt.atheism       0.81      1.00      0.89       491
           comp.graphics       0.91      0.98      0.94       504
 comp.os.ms-windows.misc       0.97      0.95      0.96       382
comp.sys.ibm.pc.hardware       0.90      0.98      0.94       365
   comp.sys.mac.hardware       0.99      0.97      0.98       369
          comp.windows.x       0.99      0.92      0.96       378
            misc.forsale       0.97      0.91      0.94       377
               rec.autos       0.98      0.99      0.99       389
         rec.motorcycles       1.00      0.99      0.99       410
      rec.sport.baseball       1.00      0.99      1.00       428
        rec.sport.hockey       0.99      1.00      1.00       449
               sci.crypt       0.99      0.99      0.99       422
         sci.electronics       0.97      0.96      0.97       378
                 sci.med       1.00      0.9